# Holomorphic Equilibrium Propagation

This notebook extends the vanilla EP demo to holomorphic EP (hEP).
The key idea: by making $\beta$ complex and evaluating on a contour,
we get **exact gradients at finite nudge amplitude** — resolving the
small-$\beta$ dilemma.

We demonstrate:
1. Why complex $\beta$ works (the Cauchy integral formula)
2. How contour integration extracts exact gradients
3. Gradient quality vs N (contour points) and r (radius)
4. Comparison with vanilla EP at various $\beta$
5. Training with hEP

Prerequisites: run `ep_demo.ipynb` first for background.

In [ ]:
using Pkg; Pkg.activate("..")
using LinearAlgebra, Statistics, Random, Plots
using Random: Xoshiro

In [ ]:
# Same network definition as vanilla EP demo
mutable struct EPNet
    W1::Matrix{Float64}
    b1::Vector{Float64}
    W2::Matrix{Float64}
    b2::Vector{Float64}
end

function EPNet(n_in, n_hid, n_out; rng=Xoshiro(42), scale=0.5)
    EPNet(
        randn(rng, n_hid, n_in) * scale,
        zeros(n_hid),
        randn(rng, n_out, n_hid) * scale,
        zeros(n_out)
    )
end

ep_cost(o, y) = 0.5 * sum(abs2.(o .- y))

## 1. Complex-Valued Settling

When $\beta$ is complex, the hidden states become complex-valued.
The activation `tanh(z)` is holomorphic (complex-differentiable),
so the settling dynamics remain well-defined.

The settling equations are identical to vanilla EP, but with complex arithmetic:

$$h \leftarrow \tanh\!\left(\tanh(W_1 x + b_1) + W_2^\top \left[\text{sech}^2(W_2 h + b_2) \odot o\right]\right)$$
$$o \leftarrow \tanh\!\left(\tanh(W_2 h + b_2) - \beta (o - y)\right)$$

In [ ]:
# Settling function that works with complex beta
function settle!(net, x, h, o, y, beta; T=100, dt=0.5)
    for t in 1:T
        pre1 = net.W1 * x .+ net.b1
        pre2 = net.W2 * h .+ net.b2
        
        grad_h = tanh.(pre1) .+ net.W2' * (sech.(pre2).^2 .* o)
        grad_o = tanh.(pre2) .- beta .* (o .- y)
        
        h .= (1 - dt) .* h .+ dt .* tanh.(grad_h)
        o .= (1 - dt) .* o .+ dt .* tanh.(grad_o)
    end
end

# Hebbian energy gradient w.r.t. W1 at given states
function hebb_W1(net, x, h)
    pre1 = net.W1 * x .+ net.b1
    return (sech.(pre1).^2 .* h) * x'
end

function hebb_W2(net, h, o)
    pre2 = net.W2 * h .+ net.b2
    return (sech.(pre2).^2 .* o) * h'
end

In [ ]:
# Verify: complex beta produces complex states
net = EPNet(2, 16, 1; scale=0.4)
x = [1.0, 0.5]
y = [1.0]

h_c = zeros(ComplexF64, 16)
o_c = zeros(ComplexF64, 1)
settle!(net, x, h_c, o_c, y, 0.5 + 0.5im; T=100)

println("Complex beta = 0.5 + 0.5i")
println("  h (first 3): ", round.(h_c[1:3], digits=4))
println("  o: ", round.(o_c, digits=4))
println("  States are complex: ", maximum(abs.(imag.(h_c))) > 1e-10)

## 2. The Contour Integration Trick

The Cauchy integral formula says: for a holomorphic function $f(\beta)$,
the derivative at $\beta=0$ is:

$$f'(0) = \frac{1}{2\pi i}\oint_{|\beta|=r} \frac{f(\beta)}{\beta^2}\,d\beta$$

Discretized with $N$ equally-spaced points on the circle
$\beta_n = r\,e^{2\pi i n/N}$:

$$f'(0) = \frac{1}{N}\sum_{n=0}^{N-1} \frac{f(\beta_n)}{\beta_n}$$

Applied to EP: $f(\beta) = \left.\frac{\partial \Phi}{\partial \theta}\right|_{z^*(\beta)}$
evaluated at the equilibrium for each $\beta_n$. The loss gradient is:

$$\frac{dL}{d\theta} = -\frac{1}{N}\sum_{n=0}^{N-1}
\left.\frac{\partial\Phi}{\partial\theta}\right|_{z^*(\beta_n)}
\cdot e^{-2\pi i n/N}$$

This is **exact for any radius $r$**, not just infinitesimal $\beta$.

In [ ]:
function hep_gradient_W1(net, x, y; N=4, r=0.5, T_free=200, T_nudge=100)
    n_hid = size(net.W1, 1)
    n_out = size(net.W2, 1)
    
    # Free phase
    h_free = zeros(n_hid)
    o_free = zeros(n_out)
    settle!(net, x, h_free, o_free, y, 0.0; T=T_free)
    
    # Contour integration
    grad_accum = zeros(ComplexF64, size(net.W1))
    
    for n in 0:N-1
        angle_n = 2π * n / N
        beta_n = r * exp(im * angle_n)
        
        # Nudge from free equilibrium
        h_n = ComplexF64.(copy(h_free))
        o_n = ComplexF64.(copy(o_free))
        settle!(net, x, h_n, o_n, y, beta_n; T=T_nudge)
        
        # Hebbian at this contour point
        hebb_n = hebb_W1(net, x, h_n)
        
        # Accumulate with inverse phase weighting
        grad_accum .+= hebb_n .* exp(-im * angle_n)
    end
    
    # Negate (dPhi/dW = -dL/dW) and take real part
    return -real.(grad_accum) ./ N
end

In [ ]:
# Finite-difference gradient (ground truth)
function fd_grad_W1(net, x, y; eps=1e-5, T=200)
    function loss_fn(W1)
        net_p = EPNet(W1, net.b1, net.W2, net.b2)
        h = zeros(size(W1, 1))
        o = zeros(size(net.W2, 1))
        settle!(net_p, x, h, o, y, 0.0; T=T)
        return ep_cost(o, y)
    end
    
    grad = zeros(size(net.W1))
    base = loss_fn(net.W1)
    for i in eachindex(net.W1)
        W1p = copy(net.W1); W1p[i] += eps
        grad[i] = (loss_fn(W1p) - base) / eps
    end
    return grad
end

## 3. Gradient Quality: hEP vs Vanilla EP

Let's compare gradient quality (cosine similarity with the true gradient)
for vanilla EP (real $\beta$) and holomorphic EP (complex contour) at
different nudge amplitudes.

In [ ]:
net = EPNet(2, 16, 1; scale=0.4)
x = [1.0, 0.5]
y = [1.0]

# Ground truth
fd = fd_grad_W1(net, x, y; T=300)
println("FD gradient norm: ", round(norm(fd), digits=6))

# Vanilla EP at various beta
ep_betas = [0.001, 0.005, 0.01, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0]
ep_cosines = Float64[]
for beta in ep_betas
    h_free = zeros(16); o_free = zeros(1)
    settle!(net, x, h_free, o_free, y, 0.0; T=300)
    hebb_f = hebb_W1(net, x, h_free)
    
    h_n = copy(h_free); o_n = copy(o_free)
    settle!(net, x, h_n, o_n, y, beta; T=150)
    hebb_n = hebb_W1(net, x, h_n)
    
    ep_g = -real.((hebb_n .- hebb_f) ./ beta)
    cs = dot(vec(ep_g), vec(fd)) / (norm(ep_g) * norm(fd) + 1e-10)
    push!(ep_cosines, cs)
end

# hEP at various r (contour radius)
hep_radii = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0]
hep_cosines = Float64[]
for r in hep_radii
    hep_g = hep_gradient_W1(net, x, y; N=8, r=r, T_free=300, T_nudge=150)
    cs = dot(vec(hep_g), vec(fd)) / (norm(hep_g) * norm(fd) + 1e-10)
    push!(hep_cosines, cs)
end

println("\nVanilla EP (real β):")
for (b, c) in zip(ep_betas, ep_cosines)
    println("  β=$(lpad(b,5))  cos=$(round(c, digits=4))")
end
println("\nhEP (complex contour, N=8):")
for (r, c) in zip(hep_radii, hep_cosines)
    println("  r=$(lpad(r,5))  cos=$(round(c, digits=4))")
end

In [ ]:
p = plot(xlabel="Nudge amplitude", ylabel="Cosine similarity",
         title="Gradient Quality: Vanilla EP vs hEP",
         xscale=:log10, ylim=(-0.5, 1.1), legend=:bottomleft)
plot!(ep_betas, ep_cosines, lw=2, marker=:circle,
      label="Vanilla EP (real β)", color=:red)
plot!(hep_radii, hep_cosines, lw=2, marker=:diamond,
      label="hEP (complex contour, N=8)", color=:blue)
hline!([1.0], ls=:dash, color=:gray, label="Perfect")
hline!([0.0], ls=:dot, color=:gray, label="Random")
p

## 4. Effect of N (Number of Contour Points)

More contour points means better rejection of higher-order terms
in the Laurent expansion. For a system where $z^*(\beta)$ is
polynomial in $\beta$ (approximately true for smooth networks),
$N$ contour points reject harmonics up to order $N-1$.

In [ ]:
r_test = 0.5
N_values = [2, 4, 8, 16, 32, 64]
n_cosines = Float64[]

for N in N_values
    hep_g = hep_gradient_W1(net, x, y; N=N, r=r_test, T_free=300, T_nudge=150)
    cs = dot(vec(hep_g), vec(fd)) / (norm(hep_g) * norm(fd) + 1e-10)
    push!(n_cosines, cs)
end

plot(N_values, n_cosines, xlabel="N (contour points)", ylabel="Cosine similarity",
     title="hEP Gradient Quality vs Contour Points (r=$r_test)",
     xscale=:log2, marker=:circle, lw=2, label="hEP",
     ylim=(0, 1.1))
hline!([1.0], ls=:dash, color=:gray, label="Perfect")

## 5. Visualizing the Contour

Let's visualize what happens in the complex $\beta$-plane.
Each contour point $\beta_n = r e^{2\pi i n/N}$ produces a
complex equilibrium. The gradient is the first Fourier coefficient
of the energy gradient evaluated at these equilibria.

In [ ]:
# Show the contour and the output at each point
N_vis = 16
r_vis = 0.5

beta_points = [r_vis * exp(im * 2π * n / N_vis) for n in 0:N_vis-1]
output_points = ComplexF64[]

# Free equilibrium first
h_free = zeros(16); o_free = zeros(1)
settle!(net, x, h_free, o_free, y, 0.0; T=300)

for beta_n in beta_points
    h_n = ComplexF64.(copy(h_free))
    o_n = ComplexF64.(copy(o_free))
    settle!(net, x, h_n, o_n, y, beta_n; T=150)
    push!(output_points, o_n[1])
end

p1 = scatter(real.(beta_points), imag.(beta_points),
    xlabel="Re(β)", ylabel="Im(β)",
    title="Contour in β-plane",
    marker=:circle, markersize=5, label="β_n",
    aspect_ratio=:equal)
scatter!([0], [0], marker=:star, markersize=8, color=:red, label="β=0")

p2 = scatter(real.(output_points), imag.(output_points),
    xlabel="Re(output)", ylabel="Im(output)",
    title="Output at each β_n",
    marker=:circle, markersize=5, label="o*(β_n)",
    aspect_ratio=:equal)
scatter!([o_free[1]], [0], marker=:star, markersize=8, color=:red, label="o*(0)")

plot(p1, p2, layout=(1,2), size=(800, 350))

## 6. Training with hEP

Now let's train the same XOR task using hEP and compare with vanilla EP.

In [ ]:
function make_xor_data(n; rng=Xoshiro(42))
    X = randn(rng, 2, n) * 0.3
    labels = [xor(x1 > 0, x2 > 0) ? 1.0 : -1.0 for (x1, x2) in eachcol(X)]
    for i in 1:n
        X[1, i] += (X[1, i] > 0 ? 0.5 : -0.5)
        X[2, i] += (X[2, i] > 0 ? 0.5 : -0.5)
    end
    return X, labels
end

X_train, y_train = make_xor_data(200)

In [ ]:
function hep_train!(net, X, Y; N=8, r=0.5, lr=0.01, epochs=30,
                    T_free=100, T_nudge=50)
    losses = Float64[]
    n_samples = length(Y)
    n_hid = size(net.W1, 1)
    n_out = size(net.W2, 1)
    
    for epoch in 1:epochs
        epoch_loss = 0.0
        for i in 1:n_samples
            xi = X[:, i]
            yi = [Y[i]]
            
            # Free phase
            h_free = zeros(n_hid)
            o_free = zeros(n_out)
            settle!(net, xi, h_free, o_free, yi, 0.0; T=T_free)
            epoch_loss += ep_cost(o_free, yi)
            
            # Contour integration
            gW1 = zeros(ComplexF64, size(net.W1))
            gW2 = zeros(ComplexF64, size(net.W2))
            gb1 = zeros(ComplexF64, size(net.b1))
            gb2 = zeros(ComplexF64, size(net.b2))
            
            for n in 0:N-1
                angle_n = 2π * n / N
                beta_n = r * exp(im * angle_n)
                
                h_n = ComplexF64.(copy(h_free))
                o_n = ComplexF64.(copy(o_free))
                settle!(net, xi, h_n, o_n, yi, beta_n; T=T_nudge)
                
                phase_w = exp(-im * angle_n)
                gW1 .+= hebb_W1(net, xi, h_n) .* phase_w
                gW2 .+= hebb_W2(net, h_n, o_n) .* phase_w
                
                pre1 = net.W1 * xi .+ net.b1
                pre2 = net.W2 * h_n .+ net.b2
                gb1 .+= (sech.(pre1).^2 .* h_n) .* phase_w
                gb2 .+= (sech.(pre2).^2 .* o_n) .* phase_w
            end
            
            # Negate and normalize
            net.W1 .+= lr .* real.(gW1) ./ N
            net.W2 .+= lr .* real.(gW2) ./ N
            net.b1 .+= lr .* real.(gb1) ./ N
            net.b2 .+= lr .* real.(gb2) ./ N
        end
        push!(losses, epoch_loss / n_samples)
    end
    return losses
end

In [ ]:
function ep_train!(net, X, Y; beta=0.1, lr=0.01, epochs=30,
                   T_free=100, T_nudge=50)
    losses = Float64[]
    n_samples = length(Y)
    n_hid = size(net.W1, 1)
    n_out = size(net.W2, 1)
    
    for epoch in 1:epochs
        epoch_loss = 0.0
        for i in 1:n_samples
            xi = X[:, i]
            yi = [Y[i]]
            
            h_free = zeros(n_hid); o_free = zeros(n_out)
            settle!(net, xi, h_free, o_free, yi, 0.0; T=T_free)
            epoch_loss += ep_cost(o_free, yi)
            hebb_f1 = hebb_W1(net, xi, h_free)
            hebb_f2 = hebb_W2(net, h_free, o_free)
            
            h_n = copy(h_free); o_n = copy(o_free)
            settle!(net, xi, h_n, o_n, yi, beta; T=T_nudge)
            hebb_n1 = hebb_W1(net, xi, h_n)
            hebb_n2 = hebb_W2(net, h_n, o_n)
            
            # EP gradient (negated for descent)
            net.W1 .+= lr .* (hebb_n1 .- hebb_f1) ./ beta
            net.W2 .+= lr .* (hebb_n2 .- hebb_f2) ./ beta
        end
        push!(losses, epoch_loss / n_samples)
    end
    return losses
end

In [ ]:
# Train both
net_ep = EPNet(2, 32, 1; rng=Xoshiro(42), scale=0.3)
net_hep = EPNet(2, 32, 1; rng=Xoshiro(42), scale=0.3)

losses_ep = ep_train!(net_ep, X_train, y_train;
    beta=0.1, lr=0.01, epochs=30, T_free=100, T_nudge=50)
losses_hep = hep_train!(net_hep, X_train, y_train;
    N=8, r=0.5, lr=0.01, epochs=30, T_free=100, T_nudge=50)

plot(losses_ep, label="Vanilla EP (β=0.1)", lw=2,
     xlabel="Epoch", ylabel="MSE Loss",
     title="EP vs hEP Training")
plot!(losses_hep, label="hEP (N=8, r=0.5)", lw=2)

In [ ]:
# Decision boundaries
function predict(net, x; T=100)
    h = zeros(size(net.W1, 1))
    o = zeros(size(net.W2, 1))
    settle!(net, x, h, o, [0.0], 0.0; T=T)
    return o[1]
end

xx = range(-1.5, 1.5, length=50)
yy = range(-1.5, 1.5, length=50)

Z_ep = [predict(net_ep, [xi, yi]) for yi in yy, xi in xx]
Z_hep = [predict(net_hep, [xi, yi]) for yi in yy, xi in xx]

p1 = contourf(xx, yy, Z_ep, levels=20, color=:RdBu, alpha=0.6,
    title="Vanilla EP")
scatter!(X_train[1,:], X_train[2,:],
    color=[(y > 0 ? :blue : :red) for y in y_train],
    markersize=2, legend=false, alpha=0.6)

p2 = contourf(xx, yy, Z_hep, levels=20, color=:RdBu, alpha=0.6,
    title="hEP")
scatter!(X_train[1,:], X_train[2,:],
    color=[(y > 0 ? :blue : :red) for y in y_train],
    markersize=2, legend=false, alpha=0.6)

plot(p1, p2, layout=(1,2), size=(800, 350))

## 7. Key Takeaways

1. **hEP gives exact gradients at finite nudge amplitude.** Unlike vanilla EP,
   where gradient quality degrades with large $\beta$, hEP maintains high
   cosine similarity with the true gradient across a wide range of $r$ values.

2. **The contour integration is a Fourier transform.** It extracts the linear
   response to the nudge (first Fourier coefficient), filtering out higher-order
   contamination that biases vanilla EP at finite $\beta$.

3. **N=8 contour points is typically sufficient.** More points improve rejection
   of higher harmonics but with diminishing returns.

4. **The physical interpretation: epicycles.** Each neuron's state traces an
   epicyclic path in the complex plane. The gradient is encoded in the amplitude
   and phase of the teaching-frequency component.

5. **Cost: N forward passes instead of one backward pass.** hEP trades the
   backward pass for N additional forward passes (one per contour point),
   making it suitable for analog/neuromorphic hardware where forward passes
   are cheap but backpropagation is impossible.